# TF-IDF + XGBoost Pipeline (Single Config)
Simple pipeline for fake news detection with default hyperparameters

In [57]:
# ============================================================================
# IMPORTS
# ============================================================================
from __future__ import annotations

import json
import re
import shutil
from pathlib import Path
from typing import Dict, List, Sequence, Tuple
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.sparse import csr_matrix, hstack
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

import torch
from transformers import RobertaTokenizer, RobertaModel
from tqdm import tqdm

sns.set_theme(style="whitegrid")
print("✅ All imports successful!")

✅ All imports successful!


In [58]:
# ============================================================================
# CONSTANTS & SETUP
# ============================================================================
RANDOM_STATE = 42

from pathlib import Path

DATA_PATH = Path("/kaggle/input/datasets/ltrungphong/dataset-123/data/full_data.csv")

RESULTS_DIR = PROJECT_ROOT / "results"
PLOTS_DIR = PROJECT_ROOT / "plots"
CM_DIR = PROJECT_ROOT / "confusion_matrix"
MODELS_DIR = PROJECT_ROOT / "models"

def ensure_output_dirs() -> None:
    for directory in [RESULTS_DIR, PLOTS_DIR, CM_DIR, MODELS_DIR]:
        directory.mkdir(parents=True, exist_ok=True)

ensure_output_dirs()
print(f"Project root: {PROJECT_ROOT}")
print(f"Output directories created ✅")

Project root: /kaggle/working
Output directories created ✅


In [59]:
# ============================================================================
# DATA PREPROCESSING
# ============================================================================
def normalize_schema(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rename_map = {}
    if "text" in df.columns and "content" not in df.columns:
        rename_map["text"] = "content"
    if "tweet_id" in df.columns and "id" not in df.columns:
        rename_map["tweet_id"] = "id"
    if rename_map:
        df = df.rename(columns=rename_map)
    if "content" not in df.columns:
        raise ValueError("Dataset must contain a text column named 'content' or 'text'.")
    if "label" not in df.columns:
        raise ValueError("Dataset must contain a 'label' column.")
    return df


def canonical_label(value: object) -> str:
    text = str(value).strip().lower()
    text = re.sub(r"[\s_\-]+", "", text)
    return text


def encode_labels(labels: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(labels):
        numeric = pd.to_numeric(labels, errors="coerce")
        valid = set(numeric.dropna().unique().tolist())
        if valid.issubset({0, 1}):
            return numeric.astype(int)

    mapping = {
        "nonrumor": 0,
        "truth": 0,
        "true": 0,
        "real": 0,
        "legit": 0,
        "legitimate": 0,
        "false": 1,
        "rumor": 1,
        "fake": 1,
        "unverified": 1,
    }
    encoded = labels.map(lambda x: mapping.get(canonical_label(x)))
    if encoded.isna().any():
        invalid = sorted({str(v) for v in labels[encoded.isna()].unique().tolist()})
        raise ValueError(
            f"Unsupported labels found. Expected values similar to true/non-rumor/false/unverified. "
            f"Invalid values: {invalid}"
        )
    return encoded.astype(int)


def clean_text(text: object) -> str:
    if pd.isna(text):
        return ""
    value = str(text).lower()
    value = re.sub(r"http\S+|www\S+|https\S+", " ", value)
    value = re.sub(r"@[A-Za-z0-9_]+", " ", value)
    value = re.sub(r"#[A-Za-z0-9_]+", " ", value)
    value = re.sub(r"[^a-z\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def extract_engineered_features(texts: Sequence[object]) -> np.ndarray:
    rows: List[List[float]] = []
    for raw_text in texts:
        value = "" if pd.isna(raw_text) else str(raw_text)
        word_count = len(value.split())
        upper_alpha = sum(1 for ch in value if ch.isalpha() and ch.isupper())
        alpha_count = sum(1 for ch in value if ch.isalpha())
        uppercase_ratio = (upper_alpha / alpha_count) if alpha_count else 0.0
        rows.append(
            [
                float(len(value)),
                float(word_count),
                float(value.count("!")),
                float(value.count("?")),
                float(uppercase_ratio),
            ]
        )
    return np.asarray(rows, dtype=np.float32)

print("✅ Data preprocessing functions defined")

✅ Data preprocessing functions defined


In [60]:
# ============================================================================
# ROBERTA SETUP
# ============================================================================
print("Setting up RoBERTa model...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
roberta_model = RobertaModel.from_pretrained("roberta-base")
roberta_model.to(device)
roberta_model.eval()

print("✅ RoBERTa model loaded successfully!")


def get_roberta_embedding(text):
    """Get [CLS] token embedding from RoBERTa (better for classification)."""
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = roberta_model(**inputs)
    # Use [CLS] token (index 0) instead of mean pooling
    return outputs.last_hidden_state[:, 0, :].cpu().numpy().flatten()


def extract_embeddings(texts):
    """Extract embeddings for a list of texts."""
    return np.vstack([get_roberta_embedding(text) for text in tqdm(texts, desc="Extracting RoBERTa embeddings")])

Setting up RoBERTa model...
Using device: cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ RoBERTa model loaded successfully!


In [61]:
# ============================================================================
# FEATURE ENGINEERING
# ============================================================================
class TextFeatureBuilder(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        max_features: int = 768,
        min_df: int = 2,
        max_df: float = 0.9,
        ngram_range: Tuple[int, int] = (1, 2),
    ) -> None:
        self.max_features = max_features
        self.min_df = min_df
        self.max_df = max_df
        self.ngram_range = ngram_range
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            min_df=min_df,
            max_df=max_df,
            ngram_range=ngram_range,
            stop_words="english",
        )

    def fit(self, X: Sequence[object], y: Sequence[int] | None = None):
        raw_texts = ["" if pd.isna(text) else str(text) for text in X]
        cleaned_texts = [clean_text(text) for text in raw_texts]
        self.vectorizer.fit(cleaned_texts)
        return self

    def transform(self, X: Sequence[object]):
        raw_texts = ["" if pd.isna(text) else str(text) for text in X]
        cleaned_texts = [clean_text(text) for text in raw_texts]
        tfidf_matrix = self.vectorizer.transform(cleaned_texts)
        engineered = csr_matrix(extract_engineered_features(raw_texts))
        return hstack([tfidf_matrix, engineered], format="csr")

    def get_feature_names_out(self) -> np.ndarray:
        tfidf_names = self.vectorizer.get_feature_names_out()
        return np.concatenate([tfidf_names, np.asarray(ENGINEERED_FEATURE_NAMES, dtype=object)])

print("✅ TextFeatureBuilder class defined")

✅ TextFeatureBuilder class defined


In [62]:
# ============================================================================
# LOAD DATASET
# ============================================================================
print("\n" + "="*80)
print("LOADING DATA")
print("="*80)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} total records")

df = normalize_schema(df)
df = df[["content", "label"]].copy()
df["content"] = df["content"].fillna("").astype(str)
df["label"] = encode_labels(df["label"])

print(f"Label distribution:")
print(df["label"].value_counts().sort_index())


LOADING DATA
Loaded 2139 total records
Label distribution:
label
0    1158
1     981
Name: count, dtype: int64


In [63]:
# ============================================================================
# TRAIN-TEST SPLIT (DEFAULT CONFIG: 80/20)
# ============================================================================
print("\n" + "="*80)
print("TRAIN-TEST SPLIT (80/20)")
print("="*80)

X = df["content"]
y = df["label"].to_numpy(dtype=int)

test_size = 0.20  # 80/20 split (keep 80% for train)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y
)

print(f"Train set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Train label distribution:\n{pd.Series(y_train).value_counts().sort_index()}")
print(f"Test label distribution:\n{pd.Series(y_test).value_counts().sort_index()}")


TRAIN-TEST SPLIT (80/20)
Train set size: 1711
Test set size: 428
Train label distribution:
0    926
1    785
Name: count, dtype: int64
Test label distribution:
0    232
1    196
Name: count, dtype: int64


In [64]:
# ============================================================================
# EXTRACT FEATURES (RoBERTa + ENGINEERED - TF-IDF will be extracted per fold)
# ============================================================================
print("\n" + "="*80)
print("FEATURE EXTRACTION (RoBERTa + Engineered)")
print("="*80)

# Extract RoBERTa embeddings (no hyperparameters, extract once in outer loop)
print("\n→ Extracting RoBERTa embeddings (TRAIN)...")
X_train_roberta = extract_embeddings(X_train.values)
print(f"  RoBERTa shape: {X_train_roberta.shape}")

print("→ Extracting RoBERTa embeddings (TEST)...")
X_test_roberta = extract_embeddings(X_test.values)
print(f"  RoBERTa shape: {X_test_roberta.shape}")

# Scale RoBERTa embeddings (fit on train only)
print("\n→ Scaling RoBERTa embeddings...")
roberta_scaler = StandardScaler()
roberta_scaler.fit(X_train_roberta)
X_train_roberta_scaled = roberta_scaler.transform(X_train_roberta)
X_test_roberta_scaled = roberta_scaler.transform(X_test_roberta)

# Extract engineered features
print("→ Extracting engineered features (TRAIN)...")
X_train_engineered = extract_engineered_features(X_train)
print(f"  Engineered shape: {X_train_engineered.shape}")

print("→ Extracting engineered features (TEST)...")
X_test_engineered = extract_engineered_features(X_test)
print(f"  Engineered shape: {X_test_engineered.shape}")

print(f"\n✅ Pre-extracted features (RoBERTa + Engineered) ready for CV!")


FEATURE EXTRACTION (RoBERTa + Engineered)

→ Extracting RoBERTa embeddings (TRAIN)...


Extracting RoBERTa embeddings: 100%|██████████| 1711/1711 [00:13<00:00, 123.92it/s]


  RoBERTa shape: (1711, 768)
→ Extracting RoBERTa embeddings (TEST)...


Extracting RoBERTa embeddings: 100%|██████████| 428/428 [00:03<00:00, 124.38it/s]

  RoBERTa shape: (428, 768)

→ Scaling RoBERTa embeddings...
→ Extracting engineered features (TRAIN)...
  Engineered shape: (1711, 5)
→ Extracting engineered features (TEST)...
  Engineered shape: (428, 5)

✅ Pre-extracted features (RoBERTa + Engineered) ready for CV!


In [65]:
# ============================================================================
# TRAIN XGBOOST (DEFAULT CONFIG)
# ============================================================================
print("\n" + "="*80)
print("TRAINING XGBOOST")
print("="*80)

# ============================================================================
# CROSS-VALIDATION WITH HYPERPARAMETER GRID SEARCH
# ============================================================================
print("\n" + "="*80)
print("CROSS-VALIDATION WITH HYPERPARAMETER GRID SEARCH (5-Fold CV)")
print("="*80)

from sklearn.model_selection import KFold
from itertools import product

# Define hyperparameter grids
tfidf_params_grid = {
    "max_features": [768, 1536, 2046],
    "min_df": [3, 4, 5, 6],
}

xgboost_params_grid = {
    "n_estimators": [300, 500],
    "learning_rate": [0.03, 0.07],
    "max_depth": [3, 5, 8],
}

# Generate all combinations
tfidf_combinations = list(product(*tfidf_params_grid.values()))
xgboost_combinations = list(product(*xgboost_params_grid.values()))
tfidf_keys = list(tfidf_params_grid.keys())
xgboost_keys = list(xgboost_params_grid.keys())

print(f"\nTF-IDF combinations: {len(tfidf_combinations)}")
print(f"XGBoost combinations: {len(xgboost_combinations)}")
print(f"Total combinations per fold: {len(tfidf_combinations) * len(xgboost_combinations)}")
print(f"Total combinations (5 folds): {5 * len(tfidf_combinations) * len(xgboost_combinations)}\n")

# Initialize KFold (5 folds)
kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Track results
cv_results = []
best_score = -np.inf
best_tfidf_params = None
best_xgboost_params = None
best_model = None

# Total combinations
total_combos = 5 * len(tfidf_combinations) * len(xgboost_combinations)
combo_counter = 0

# OUTER LOOP: K-Fold splits
fold_counter = 0
for train_idx, val_idx in kfold.split(X_train):
    fold_counter += 1
    print(f"\n{'='*80}")
    print(f"FOLD {fold_counter}/5")
    print(f"{'='*80}")
    
    # Get fold data
    X_train_fold_all = X_train.iloc[train_idx].values
    X_val_fold_all = X_train.iloc[val_idx].values
    y_train_fold = y_train[train_idx]
    y_val_fold = y_train[val_idx]
    
    # Extract RoBERTa embeddings ONCE per fold (no hyperparameters)
    print(f"\n→ Extracting RoBERTa embeddings for fold {fold_counter}...")
    X_train_roberta_fold = extract_embeddings(X_train_fold_all)
    X_val_roberta_fold = extract_embeddings(X_val_fold_all)
    print(f"  Train RoBERTa shape: {X_train_roberta_fold.shape}")
    print(f"  Val RoBERTa shape: {X_val_roberta_fold.shape}")
    
    # Scale RoBERTa for this fold
    roberta_scaler_fold = StandardScaler()
    roberta_scaler_fold.fit(X_train_roberta_fold)
    X_train_roberta_scaled_fold = roberta_scaler_fold.transform(X_train_roberta_fold)
    X_val_roberta_scaled_fold = roberta_scaler_fold.transform(X_val_roberta_fold)
    
    # Extract engineered features for this fold
    print(f"→ Extracting engineered features for fold {fold_counter}...")
    X_train_eng_fold = extract_engineered_features(X_train_fold_all)
    X_val_eng_fold = extract_engineered_features(X_val_fold_all)
    print(f"  Train engineered shape: {X_train_eng_fold.shape}")
    print(f"  Val engineered shape: {X_val_eng_fold.shape}")
    
    # MIDDLE LOOP: TF-IDF hyperparameters
    for tfidf_combo in tfidf_combinations:
        tfidf_params = dict(zip(tfidf_keys, tfidf_combo))
        
        # INNER LOOP: XGBoost hyperparameters
        for xgb_combo in xgboost_combinations:
            combo_counter += 1
            xgboost_params = dict(zip(xgboost_keys, xgb_combo))
            
            # Progress display
            progress_str = f"[{combo_counter:4d}/{total_combos}] (Fold {fold_counter}/5)"
            param_str = f"TF-IDF: max_feat={tfidf_params['max_features']}, min_df={tfidf_params['min_df']} | XGB: n_est={xgboost_params['n_estimators']}, lr={xgboost_params['learning_rate']}, depth={xgboost_params['max_depth']}"
            print(f"\n{progress_str} {param_str}")
            
            # Extract TF-IDF with current hyperparameters for this fold
            fold_builder = TextFeatureBuilder(
                max_features=tfidf_params["max_features"],
                min_df=tfidf_params["min_df"],
                max_df=0.9,
                ngram_range=(1, 2),
            )
            fold_builder.fit(X_train_fold_all)
            
            X_train_tfidf_fold = fold_builder.transform(X_train_fold_all).toarray()
            X_val_tfidf_fold = fold_builder.transform(X_val_fold_all).toarray()
            
            # Combine features (TF-IDF + RoBERTa + Engineered)
            X_train_combined = np.hstack([X_train_tfidf_fold, X_train_roberta_scaled_fold, X_train_eng_fold])
            X_val_combined = np.hstack([X_val_tfidf_fold, X_val_roberta_scaled_fold, X_val_eng_fold])
            
            # Train XGBoost model
            cv_model = XGBClassifier(
                n_estimators=xgboost_params["n_estimators"],
                max_depth=xgboost_params["max_depth"],
                learning_rate=xgboost_params["learning_rate"],
                eval_metric="logloss",
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                importance_type="gain",
                device='cuda',
                max_bin=256,
            )
            cv_model.fit(X_train_combined, y_train_fold, verbose=False)
            
            # Evaluate on validation fold
            y_val_pred_proba = cv_model.predict_proba(X_val_combined)[:, 1]
            y_val_pred = (y_val_pred_proba >= 0.5).astype(int)
            
            fold_f1 = f1_score(y_val_fold, y_val_pred, zero_division=0)
            fold_acc = accuracy_score(y_val_fold, y_val_pred)
            
            print(f"   → F1: {fold_f1:.4f}, Accuracy: {fold_acc:.4f}")
            
            # Store result
            cv_results.append({
                "fold": fold_counter,
                "tfidf_params": tfidf_params,
                "xgboost_params": xgboost_params,
                "fold_f1": fold_f1,
                "fold_acc": fold_acc,
            })

# ============================================================================
# FIND BEST CONFIG BASED ON AVERAGE SCORE ACROSS 5 FOLDS
# ============================================================================
print("\n" + "="*80)
print("AGGREGATING CV RESULTS (Average across 5 folds)")
print("="*80)

# Group results by config
config_results = {}
for result in cv_results:
    tfidf_key = str(result["tfidf_params"])
    xgb_key = str(result["xgboost_params"])
    config_key = (tfidf_key, xgb_key)
    
    if config_key not in config_results:
        config_results[config_key] = {
            "tfidf_params": result["tfidf_params"],
            "xgboost_params": result["xgboost_params"],
            "fold_f1_scores": [],
            "fold_acc_scores": [],
        }
    
    config_results[config_key]["fold_f1_scores"].append(result["fold_f1"])
    config_results[config_key]["fold_acc_scores"].append(result["fold_acc"])

# Compute averages and find best
print(f"\nTotal unique configurations tested: {len(config_results)}\n")

best_configs = []
for config_key, results in config_results.items():
    avg_f1 = np.mean(results["fold_f1_scores"])
    avg_acc = np.mean(results["fold_acc_scores"])
    std_f1 = np.std(results["fold_f1_scores"])
    std_acc = np.std(results["fold_acc_scores"])
    
    best_configs.append({
        "tfidf_params": results["tfidf_params"],
        "xgboost_params": results["xgboost_params"],
        "avg_f1": avg_f1,
        "std_f1": std_f1,
        "avg_acc": avg_acc,
        "std_acc": std_acc,
        "fold_f1_scores": results["fold_f1_scores"],
        "fold_acc_scores": results["fold_acc_scores"],
    })

# Sort by average F1 score
best_configs.sort(key=lambda x: x["avg_f1"], reverse=True)

# Display top 5 best configurations
print("Top 5 Best Configurations (by average F1 across 5 folds):\n")
for rank, config in enumerate(best_configs[:5], 1):
    print(f"{rank}. TF-IDF: {config['tfidf_params']} | XGBoost: {config['xgboost_params']}")
    print(f"   Avg F1:  {config['avg_f1']:.4f} ± {config['std_f1']:.4f}")
    print(f"   Avg Acc: {config['avg_acc']:.4f} ± {config['std_acc']:.4f}")
    print(f"   Fold F1 scores: {[f'{s:.4f}' for s in config['fold_f1_scores']]}")
    print()

# Get best configuration
best_config = best_configs[0]
best_tfidf_params = best_config["tfidf_params"]
best_xgboost_params = best_config["xgboost_params"]
best_score = best_config["avg_f1"]

# Display best result
print("="*80)
print("BEST HYPERPARAMETERS (FROM 5-FOLD CV - Average)")
print("="*80)
print(f"\nBest Average F1 Score: {best_score:.4f} ± {best_config['std_f1']:.4f}")
print(f"Best Average Accuracy: {best_config['avg_acc']:.4f} ± {best_config['std_acc']:.4f}")
print(f"Best TF-IDF params: {best_tfidf_params}")
print(f"Best XGBoost params: {best_xgboost_params}")

# Retrain on full training set with best hyperparameters
print("\n→ Retraining on full training set with best hyperparameters...")

# Extract TF-IDF on full training set with best params
feature_builder = TextFeatureBuilder(
    max_features=best_tfidf_params["max_features"],
    min_df=best_tfidf_params["min_df"],
    max_df=0.9,
    ngram_range=(1, 2),
)
feature_builder.fit(X_train)
X_train_tfidf_final = feature_builder.transform(X_train).toarray()
X_test_tfidf_final = feature_builder.transform(X_test).toarray()

# Combine all features for final model
X_train_features_final = np.hstack([X_train_tfidf_final, X_train_roberta_scaled, X_train_engineered])
X_test_features_final = np.hstack([X_test_tfidf_final, X_test_roberta_scaled, X_test_engineered])

# Split for loss tracking
X_train_fit, X_val, y_train_fit, y_val = train_test_split(
    X_train_features_final, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
)

# Train final model with eval tracking
model = XGBClassifier(
    n_estimators=best_xgboost_params["n_estimators"],
    max_depth=best_xgboost_params["max_depth"],
    learning_rate=best_xgboost_params["learning_rate"],
    eval_metric=["logloss", "error"],
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    importance_type="gain",
    device='cuda',
    max_bin=256,
)

# Train with eval_set to track loss and accuracy
eval_set = [(X_train_fit, y_train_fit), (X_val, y_val)]
model.fit(
    X_train_fit, 
    y_train_fit,
    eval_set=eval_set,
    verbose=False
)

# Store loss and accuracy history for visualization
results = model.evals_result()
train_loss = results['validation_0']['logloss']
val_loss = results['validation_1']['logloss']
train_error = results['validation_0']['error']
val_error = results['validation_1']['error']

# Convert error to accuracy (accuracy = 1 - error)
train_accuracy = [1 - e for e in train_error]
val_accuracy = [1 - e for e in val_error]

# Retrain on full training set (without eval_set) for final model
print("→ Final retraining on full training set (no validation split)...")
model_final = XGBClassifier(
    n_estimators=best_xgboost_params["n_estimators"],
    max_depth=best_xgboost_params["max_depth"],
    learning_rate=best_xgboost_params["learning_rate"],
    eval_metric="logloss",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    importance_type="gain",
    device='cuda',
    max_bin=256,
)
model_final.fit(X_train_features_final, y_train, verbose=False)
model = model_final

# Store vectorizer for later use
tfidf_vec = feature_builder.vectorizer
roberta_scaler_final = roberta_scaler

print("✅ Training complete!")


TRAINING XGBOOST

CROSS-VALIDATION WITH HYPERPARAMETER GRID SEARCH (5-Fold CV)

TF-IDF combinations: 12
XGBoost combinations: 12
Total combinations per fold: 144
Total combinations (5 folds): 720


FOLD 1/5

→ Extracting RoBERTa embeddings for fold 1...


Extracting RoBERTa embeddings: 100%|██████████| 343/343 [00:02<00:00, 125.20it/s]


  Train RoBERTa shape: (1368, 768)
  Val RoBERTa shape: (343, 768)
→ Extracting engineered features for fold 1...
  Train engineered shape: (1368, 5)
  Val engineered shape: (343, 5)

[   1/720] (Fold 1/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.03, depth=3


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [16:32:20] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


   → F1: 0.7668, Accuracy: 0.7872

[   2/720] (Fold 1/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.03, depth=5
   → F1: 0.7683, Accuracy: 0.7872

[   3/720] (Fold 1/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.03, depth=8
   → F1: 0.7468, Accuracy: 0.7726

[   4/720] (Fold 1/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.07, depth=3
   → F1: 0.7539, Accuracy: 0.7697

[   5/720] (Fold 1/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.07, depth=5
   → F1: 0.7500, Accuracy: 0.7726

[   6/720] (Fold 1/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.07, depth=8
   → F1: 0.7625, Accuracy: 0.7930

[   7/720] (Fold 1/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=500, lr=0.03, depth=3
   → F1: 0.7524, Accuracy: 0.7697

[   8/720] (Fold 1/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=500, lr=0.03, depth=5
   → F1: 0.7604, Accuracy: 0.7813

[   9/720] (Fold 1/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=500, lr=0.03, depth=8
   → F1: 0.7588,

...

  Train RoBERTa shape: (1369, 768)
  Val RoBERTa shape: (342, 768)
→ Extracting engineered features for fold 3...
  Train engineered shape: (1369, 5)
  Val engineered shape: (342, 5)

[ 289/720] (Fold 3/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.03, depth=3
   → F1: 0.7302, Accuracy: 0.7515

[ 290/720] (Fold 3/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.03, depth=5
   → F1: 0.7460, Accuracy: 0.7690

[ 291/720] (Fold 3/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.03, depth=8
   → F1: 0.7434, Accuracy: 0.7719

[ 292/720] (Fold 3/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.07, depth=3
   → F1: 0.7546, Accuracy: 0.7661

[ 293/720] (Fold 3/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.07, depth=5
   → F1: 0.7717, Accuracy: 0.7924

[ 294/720] (Fold 3/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.07, depth=8
   → F1: 0.7314, Accuracy: 0.7573

[ 295/720] (Fold 3/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=500, lr=0.03

Extracting RoBERTa embeddings: 100%|██████████| 342/342 [00:02<00:00, 125.97it/s]


  Train RoBERTa shape: (1369, 768)
  Val RoBERTa shape: (342, 768)
→ Extracting engineered features for fold 4...
  Train engineered shape: (1369, 5)
  Val engineered shape: (342, 5)

[ 433/720] (Fold 4/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.03, depth=3
   → F1: 0.6797, Accuracy: 0.7135

[ 434/720] (Fold 4/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.03, depth=5
   → F1: 0.7308, Accuracy: 0.7544

[ 435/720] (Fold 4/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.03, depth=8
   → F1: 0.6948, Accuracy: 0.7251

[ 436/720] (Fold 4/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.07, depth=3
   → F1: 0.6903, Accuracy: 0.7193

[ 437/720] (Fold 4/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.07, depth=5
   → F1: 0.7290, Accuracy: 0.7544

[ 438/720] (Fold 4/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.07, depth=8
   → F1: 0.7032, Accuracy: 0.7310

[ 439/720] (Fold 4/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=500, lr=0.03

Extracting RoBERTa embeddings: 100%|██████████| 342/342 [00:02<00:00, 127.10it/s]


  Train RoBERTa shape: (1369, 768)
  Val RoBERTa shape: (342, 768)
→ Extracting engineered features for fold 5...
  Train engineered shape: (1369, 5)
  Val engineered shape: (342, 5)

[ 577/720] (Fold 5/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.03, depth=3
   → F1: 0.7571, Accuracy: 0.8012

[ 578/720] (Fold 5/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.03, depth=5
   → F1: 0.7500, Accuracy: 0.7953

[ 579/720] (Fold 5/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.03, depth=8
   → F1: 0.7698, Accuracy: 0.8129

[ 580/720] (Fold 5/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.07, depth=3
   → F1: 0.7730, Accuracy: 0.8129

[ 581/720] (Fold 5/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.07, depth=5
   → F1: 0.7692, Accuracy: 0.8158

[ 582/720] (Fold 5/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=300, lr=0.07, depth=8
   → F1: 0.7518, Accuracy: 0.8012

[ 583/720] (Fold 5/5) TF-IDF: max_feat=768, min_df=3 | XGB: n_est=500, lr=0.03

In [66]:
# ============================================================================
# EVALUATE ON TEST SET
# ============================================================================
print("\n" + "="*80)
print("EVALUATION")
print("="*80)

y_prob = model.predict_proba(X_test_features_final)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

# Compute metrics
test_accuracy = accuracy_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred, zero_division=0)
test_recall = recall_score(y_test, y_pred, zero_division=0)
test_f1 = f1_score(y_test, y_pred, zero_division=0)
test_roc_auc = roc_auc_score(y_test, y_prob)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]

print(f"\n📊 TEST SET METRICS:")
print(f"  Accuracy:  {test_accuracy:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall:    {test_recall:.4f}")
print(f"  F1 Score:  {test_f1:.4f}")
print(f"  ROC-AUC:   {test_roc_auc:.4f}")

print(f"\n🎯 CONFUSION MATRIX:")
print(f"     Pred 0  Pred 1")
print(f"Act 0  {tn:5d}  {fp:5d}")
print(f"Act 1  {fn:5d}  {tp:5d}")

print(f"\n📋 CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred, target_names=["Truth", "Rumor"]))


EVALUATION

📊 TEST SET METRICS:
  Accuracy:  0.8107
  Precision: 0.8010
  Recall:    0.7806
  F1 Score:  0.7907
  ROC-AUC:   0.8757

🎯 CONFUSION MATRIX:
     Pred 0  Pred 1
Act 0    194     38
Act 1     43    153

📋 CLASSIFICATION REPORT:
              precision    recall  f1-score   support

       Truth       0.82      0.84      0.83       232
       Rumor       0.80      0.78      0.79       196

    accuracy                           0.81       428
   macro avg       0.81      0.81      0.81       428
weighted avg       0.81      0.81      0.81       428



In [67]:
# ============================================================================
# VISUALIZATIONS
# ============================================================================
print("\n" + "="*80)
print("GENERATING PLOTS")
print("="*80)

# Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Truth", "Rumor"],
    yticklabels=["Truth", "Rumor"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig(CM_DIR / "confusion_matrix.png", dpi=200)
plt.close()
print(f"✅ Saved: confusion_matrix.png")

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2.5, label=f"ROC AUC = {test_roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.5, label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "roc_curve.png", dpi=200)
plt.close()
print(f"✅ Saved: roc_curve.png")


GENERATING PLOTS
✅ Saved: confusion_matrix.png
✅ Saved: roc_curve.png


In [68]:
# Loss Curve
plt.figure(figsize=(10, 6))
epochs = range(1, len(train_loss) + 1)
plt.plot(epochs, train_loss, linewidth=2, label="Train Loss (LogLoss)", marker='o', markersize=3, alpha=0.7)
plt.plot(epochs, val_loss, linewidth=2, label="Validation Loss (LogLoss)", marker='s', markersize=3, alpha=0.7)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "loss_curve.png", dpi=200)
plt.close()
print(f"✅ Saved: loss_curve.png")

✅ Saved: loss_curve.png


In [69]:
# ============================================================================
# FEATURE IMPORTANCE
# ============================================================================
print("\nExtracting feature importance...")

# Get feature names from TextFeatureBuilder
all_feature_names_from_builder = feature_builder.get_feature_names_out()

# Separate TF-IDF feature names and engineered feature names
n_tfidf_features = len(all_feature_names_from_builder) - len(ENGINEERED_FEATURE_NAMES)
tfidf_feature_names = all_feature_names_from_builder[:n_tfidf_features]
engineered_feature_names = all_feature_names_from_builder[n_tfidf_features:]

# Generate RoBERTa feature names
roberta_feature_names = np.array([f"RoBERTa_dim_{i}" for i in range(X_train_roberta_scaled.shape[1])])

# Combine all feature names
all_feature_names = np.concatenate([
    tfidf_feature_names,
    roberta_feature_names,
    engineered_feature_names
])

importances = np.asarray(model.feature_importances_, dtype=float)

top_k = min(20, importances.size)
top_indices = np.argsort(importances)[::-1][:top_k]
top_features = all_feature_names[top_indices]
top_values = importances[top_indices]

# Plot Feature Importance
plt.figure(figsize=(10, 7))
sns.barplot(x=top_values, y=top_features, palette="viridis")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 20 Feature Importances (TF-IDF + RoBERTa + Engineered)")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "feature_importance.png", dpi=200)
plt.close()
print(f"✅ Saved: feature_importance.png")

# Print top features
print(f"\nTop 20 Features by Importance:")
for rank, (feat, val) in enumerate(zip(top_features, top_values), 1):
    print(f"  {rank:2d}. {feat:40s} → {val:.6f}")


Extracting feature importance...


/tmp/ipykernel_55/3082749400.py:33: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_values, y=top_features, palette="viridis")


✅ Saved: feature_importance.png

Top 20 Features by Importance:
   1. RoBERTa_dim_621                          → 0.009801
   2. walker                                   → 0.009104
   3. transgender                              → 0.008783
   4. white                                    → 0.007308
   5. RoBERTa_dim_620                          → 0.006560
   6. RoBERTa_dim_100                          → 0.006401
   7. RoBERTa_dim_542                          → 0.006255
   8. RoBERTa_dim_420                          → 0.005956
   9. RoBERTa_dim_426                          → 0.005883
  10. paul                                     → 0.005367
  11. RoBERTa_dim_622                          → 0.005101
  12. RoBERTa_dim_609                          → 0.004994
  13. RoBERTa_dim_141                          → 0.004690
  14. RoBERTa_dim_82                           → 0.004554
  15. RoBERTa_dim_10                           → 0.004547
  16. RoBERTa_dim_465                          → 0.004521
  17. Ro

In [70]:
# Accuracy Curve
plt.figure(figsize=(10, 6))
epochs = range(1, len(train_accuracy) + 1)
plt.plot(epochs, train_accuracy, linewidth=2, label="Train Accuracy", marker='o', markersize=3, alpha=0.7)
plt.plot(epochs, val_accuracy, linewidth=2, label="Validation Accuracy", marker='s', markersize=3, alpha=0.7)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.ylim([0, 1])
plt.tight_layout()
plt.savefig(PLOTS_DIR / "accuracy_curve.png", dpi=200)
plt.close()
print(f"✅ Saved: accuracy_curve.png")

✅ Saved: accuracy_curve.png


In [71]:
# ============================================================================
# SAVE RESULTS & MODELS
# ============================================================================
print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)

# Results JSON
results = {
    "timestamp": datetime.now().isoformat(),
    "cv_selection": "Average across 5 folds",
    "cv_best_tfidf_params": best_tfidf_params,
    "cv_best_xgboost_params": best_xgboost_params,
    "cv_metrics": {
        "best_avg_f1_score": float(best_config["avg_f1"]),
        "best_f1_std": float(best_config["std_f1"]),
        "best_avg_accuracy": float(best_config["avg_acc"]),
        "best_acc_std": float(best_config["std_acc"]),
        "fold_f1_scores": [float(s) for s in best_config["fold_f1_scores"]],
        "fold_acc_scores": [float(s) for s in best_config["fold_acc_scores"]],
    },
    "split_ratio": "80/20 with 5-Fold CV",
    "features": "TF-IDF + RoBERTa (768) + Engineered (5)",
    "train_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
    "test_metrics": {
        "accuracy": float(test_accuracy),
        "precision": float(test_precision),
        "recall": float(test_recall),
        "f1_score": float(test_f1),
        "roc_auc": float(test_roc_auc),
    },
    "confusion_matrix": {
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
    },
}

with open(RESULTS_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print("✅ Saved: results.json")

# Save model
joblib.dump(model, MODELS_DIR / "xgboost_model.joblib")
print("✅ Saved: xgboost_model.joblib")

# Save TF-IDF vectorizer
joblib.dump(tfidf_vec, MODELS_DIR / "tfidf_vectorizer.joblib")
print("✅ Saved: tfidf_vectorizer.joblib")

# Save scaler
joblib.dump(roberta_scaler_final, MODELS_DIR / "roberta_scaler.joblib")
print("✅ Saved: roberta_scaler.joblib")

print("\n" + "="*80)
print("✅ PIPELINE COMPLETE!")
print("="*80)
print(f"\nResults saved to: {RESULTS_DIR}")
print(f"Plots saved to: {PLOTS_DIR}")
print(f"Models saved to: {MODELS_DIR}")


SAVING RESULTS
✅ Saved: results.json
✅ Saved: xgboost_model.joblib
✅ Saved: tfidf_vectorizer.joblib
✅ Saved: roberta_scaler.joblib

✅ PIPELINE COMPLETE!

Results saved to: /kaggle/working/results
Plots saved to: /kaggle/working/plots
Models saved to: /kaggle/working/models


In [72]:
# ============================================================================
# SUMMARY
# ============================================================================
print("\n📋 FINAL SUMMARY")
print("="*80)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")
print(f"ROC-AUC Score: {test_roc_auc:.4f}")
print("="*80)


📋 FINAL SUMMARY
Test Accuracy: 0.8107
Test F1 Score: 0.7907
ROC-AUC Score: 0.8757
